In [12]:
import time
import numpy as np
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer

In [13]:
short_text  = "Oh brilliant, just what I needed."
medium_text = "The service was absolutely fantastic, I would recommend to everyone."
long_text = "The service was absolutely fantastic and I cannot begin to express how thoroughly impressed I was with every single aspect of my visit to this establishment, from the moment I walked through the door to the time I left, everything was handled with such professionalism and care that I would without hesitation recommend this place to absolutely everyone I know and have ever met in my entire life, it was truly a remarkable and unforgettable experience that exceeded all of my expectations completely."

In [14]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {DEVICE}")

Using: cuda


In [15]:
def time_model(predict_fn, text, n_runs=20, n_warmup=3):
    for _ in range(n_warmup):
        predict_fn(text)

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        predict_fn(text)
        end = time.perf_counter()
        times.append((end - start) * 1000)
    return {
        "mean_ms": round(np.mean(times), 2),
        "std_ms":  round(np.std(times), 2)
    }

In [16]:
model = RobertaForSequenceClassification.from_pretrained(
    "joela0/besstie-roberta-all-pool"
).to(DEVICE)
tokenizer = RobertaTokenizer.from_pretrained(
    "joela0/besstie-roberta-all-pool"
)
model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3843.37it/s]


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [17]:
def predict_roberta(text):
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=128
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    return torch.argmax(outputs.logits).item()

print("Short:", time_model(predict_roberta, short_text))
print("Medium:", time_model(predict_roberta, medium_text))
print("Long:", time_model(predict_roberta, long_text))

Short: {'mean_ms': np.float64(22.05), 'std_ms': np.float64(3.04)}
Medium: {'mean_ms': np.float64(21.93), 'std_ms': np.float64(2.43)}
Long: {'mean_ms': np.float64(19.61), 'std_ms': np.float64(2.21)}
